# Query a Database in Plain English with MCP — runnable notebook

This notebook is a Colab/Kaggle-friendly demo of the **query and schema-inspection functions** behind the
[Python & Data Analysis course](https://github.com/abderrahim-lectures/python-data-analysis-course)'s
**Query a Database in Plain English with MCP** project. It mirrors `db_tools.py` from
[`examples/mcp-sqlite-server/`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/mcp-sqlite-server/db_tools.py).

**This is not the MCP server, and it does not talk to Claude Desktop or any MCP client.** MCP servers are long-running
local processes that a desktop AI client connects to directly -- not something a hosted notebook can be. What this
notebook *does* show, in isolation and with plain function calls, is the actual logic the real server wraps as MCP
tools: building a small sample SQLite database, listing its tables, describing a table's schema, and running a
safely-checked read-only SQL query. For the real MCP server and the Claude Desktop connection, see the lesson itself
and run `examples/mcp-sqlite-server/server.py` locally.

## Build the sample database

No installs needed for this part -- `sqlite3` is in the Python standard library.

In [ ]:
import re
import sqlite3
from pathlib import Path

DB_PATH = Path("library.db")

SCHEMA = """
CREATE TABLE authors (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL,
    country TEXT
);

CREATE TABLE books (
    id         INTEGER PRIMARY KEY,
    title      TEXT NOT NULL,
    author_id  INTEGER NOT NULL REFERENCES authors(id),
    year       INTEGER,
    genre      TEXT
);

CREATE TABLE members (
    id         INTEGER PRIMARY KEY,
    name       TEXT NOT NULL,
    joined_on  TEXT NOT NULL
);

CREATE TABLE loans (
    id          INTEGER PRIMARY KEY,
    book_id     INTEGER NOT NULL REFERENCES books(id),
    member_id   INTEGER NOT NULL REFERENCES members(id),
    borrowed_on TEXT NOT NULL,
    returned_on TEXT
);
"""

AUTHORS = [
    (1, "Ursula K. Le Guin", "USA"),
    (2, "Italo Calvino", "Italy"),
    (3, "Yoko Ogawa", "Japan"),
    (4, "Chimamanda Ngozi Adichie", "Nigeria"),
    (5, "Jorge Luis Borges", "Argentina"),
]

BOOKS = [
    (1, "The Left Hand of Darkness", 1, 1969, "Science Fiction"),
    (2, "The Dispossessed", 1, 1974, "Science Fiction"),
    (3, "Invisible Cities", 2, 1972, "Fiction"),
    (4, "If on a winter\'s night a traveler", 2, 1979, "Fiction"),
    (5, "The Memory Police", 3, 1994, "Fiction"),
    (6, "Hotel Iris", 3, 1996, "Fiction"),
    (7, "Half of a Yellow Sun", 4, 2006, "Historical Fiction"),
    (8, "Americanah", 4, 2013, "Fiction"),
    (9, "Ficciones", 5, 1944, "Short Stories"),
    (10, "The Aleph", 5, 1949, "Short Stories"),
]

MEMBERS = [
    (1, "Amara Okafor", "2024-01-15"),
    (2, "Kenji Watanabe", "2024-02-03"),
    (3, "Sofia Rossi", "2024-03-21"),
    (4, "Liam O\'Connor", "2024-05-09"),
    (5, "Priya Sharma", "2024-06-30"),
]

LOANS = [
    (1, 1, 1, "2024-06-01", "2024-06-15"),
    (2, 3, 2, "2024-06-05", "2024-06-19"),
    (3, 5, 3, "2024-06-10", None),
    (4, 7, 4, "2024-06-12", "2024-06-26"),
    (5, 2, 1, "2024-07-01", None),
    (6, 9, 5, "2024-07-03", "2024-07-17"),
    (7, 4, 2, "2024-07-08", None),
    (8, 6, 3, "2024-07-10", "2024-07-24"),
    (9, 8, 4, "2024-07-12", None),
    (10, 10, 5, "2024-07-15", None),
]

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.executescript(SCHEMA)
conn.executemany("INSERT INTO authors VALUES (?, ?, ?)", AUTHORS)
conn.executemany("INSERT INTO books VALUES (?, ?, ?, ?, ?)", BOOKS)
conn.executemany("INSERT INTO members VALUES (?, ?, ?)", MEMBERS)
conn.executemany("INSERT INTO loans VALUES (?, ?, ?, ?, ?)", LOANS)
conn.commit()
conn.close()

print(f"Built sample database at {DB_PATH.resolve()}")

## The query and schema-inspection functions

These are the same three functions `server.py` wraps with `@mcp.tool()` in the real project -- here they're called
directly, no MCP involved. `run_read_only_query` is the security-relevant one: it rejects anything that isn't a
single, standalone `SELECT`, and additionally opens the database file itself in SQLite's read-only mode as a second,
independent layer of defense.

In [ ]:
_FORBIDDEN_KEYWORDS = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|ATTACH|DETACH|PRAGMA|VACUUM)\b",
    re.IGNORECASE,
)


class UnsafeQueryError(ValueError):
    """Raised when a query isn\'t a single, read-only SELECT."""


def _connect(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn


def list_tables(db_path=DB_PATH):
    """Return the names of every user table in the database."""
    conn = _connect(db_path)
    try:
        rows = conn.execute(
            "SELECT name FROM sqlite_master WHERE type = \'table\' AND name NOT LIKE \'sqlite_%\' ORDER BY name"
        ).fetchall()
        return [row["name"] for row in rows]
    finally:
        conn.close()


def describe_table(table_name, db_path=DB_PATH):
    """Return each column\'s name, declared type, and nullability for a table."""
    if table_name not in list_tables(db_path):
        raise ValueError(f"No such table: {table_name!r}. Call list_tables() to see what\'s available.")

    conn = _connect(db_path)
    try:
        rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
        return [
            {
                "name": row["name"],
                "type": row["type"],
                "nullable": not row["notnull"],
                "primary_key": bool(row["pk"]),
            }
            for row in rows
        ]
    finally:
        conn.close()


def run_read_only_query(sql, db_path=DB_PATH):
    """Run a single read-only SELECT query and return the matching rows."""
    stripped = sql.strip().rstrip(";")
    if not stripped:
        raise UnsafeQueryError("Empty query.")
    if ";" in stripped:
        raise UnsafeQueryError("Only a single statement is allowed -- no \';\' inside the query.")
    if not stripped.upper().startswith("SELECT"):
        raise UnsafeQueryError("Only SELECT queries are allowed.")
    if _FORBIDDEN_KEYWORDS.search(stripped):
        raise UnsafeQueryError("Query contains a write/DDL keyword, which isn\'t allowed.")

    uri = f"file:{Path(db_path).resolve()}?mode=ro"
    conn = sqlite3.connect(uri, uri=True)
    conn.row_factory = sqlite3.Row
    try:
        rows = conn.execute(stripped).fetchall()
        return [dict(row) for row in rows]
    finally:
        conn.close()

## Try them out

First, see what tables exist, then look at one table's shape:

In [ ]:
list_tables()

In [ ]:
describe_table("books")

Now a real read-only query -- the same shape an LLM client would compose on its own from a plain-English question:

In [ ]:
run_read_only_query("SELECT title, year FROM books WHERE genre = 'Fiction' ORDER BY year")

And the safety check actually rejecting an unsafe query, instead of quietly running it:

In [ ]:
try:
    run_read_only_query("DROP TABLE books")
except UnsafeQueryError as exc:
    print(f"Rejected, as expected: {exc}")

## What this notebook doesn't show

This notebook proves the query and schema logic works -- it does **not** start an MCP server, does **not** speak
the MCP protocol, and does **not** connect to Claude Desktop or any other MCP client. For that, follow the lesson's
own steps locally with `uv`, using `examples/mcp-sqlite-server/server.py` (which wraps these exact same functions,
imported from `db_tools.py`, as MCP tools with `@mcp.tool()`).